In [1]:
import pandas as pd
import numpy as np
import time

def generate_telemetry_dataset(filename="network_telemetry.csv", duration_minutes=60):
    np.random.seed(42)
    start_time = int(time.time())
    records = []
    
    # Define Core Network Topology Coordinates (in meters)
    towers = {
        'TOWER_A': {'x': 1000, 'y': 1000},
        'TOWER_B': {'x': 2500, 'y': 1000}
    }
    
    print(u"Generating synthetic network telemetry data...")
    
    # Generate data second-by-second
    for sec in range(duration_minutes * 60):
        current_time = start_time + sec
        
        # Check if we are in the "Traffic Jam Anomaly Zone" (Last 15 minutes)
        is_anomaly = sec > (duration_minutes - 15) * 60
        anomaly_progress = (sec - (duration_minutes - 15) * 60) / 900.0 if is_anomaly else 0.0
        
        # --- 1. SIMULATE TOWER A (The Bottleneck Target) ---
        base_users_a = np.random.randint(150, 220)
        # If anomaly hits, scale users exponentially
        surge_users_a = int(base_users_a + (anomaly_progress * 1200)) if is_anomaly else base_users_a
        
        # Calculate resulting resource utilization
        prb_util_a = min(0.35 + (surge_users_a / 1400.0) + np.random.normal(0, 0.02), 1.0)
        throughput_a = surge_users_a * np.random.uniform(0.002, 0.005) # Gbps
        
        records.append({
            'timestamp': current_time, 'tower_id': 'TOWER_A', 
            'active_users': surge_users_a, 'resource_block_util': round(prb_util_a, 4),
            'backhaul_throughput': round(throughput_a, 2)
        })
        
        # --- 2. SIMULATE TOWER B (The Safe Haven Adjacent Node) ---
        users_b = np.random.randint(100, 140)
        prb_util_b = 0.20 + (users_b / 2000.0) + np.random.normal(0, 0.01)
        throughput_b = users_b * np.random.uniform(0.002, 0.004)
        
        records.append({
            'timestamp': current_time, 'tower_id': 'TOWER_B', 
            'active_users': users_b, 'resource_block_util': round(prb_util_b, 4),
            'backhaul_throughput': round(throughput_b, 2)
        })
        
    df = pd.DataFrame(records)
    df.to_csv(filename, index=False)
    print(f"Success! Dataset saved to {filename} ({len(df)} rows)")

if __name__ == "__main__":
    generate_telemetry_dataset()

Generating synthetic network telemetry data...
Success! Dataset saved to network_telemetry.csv (7200 rows)


In [2]:
import pandas as pd
import numpy as np
import time

def generate_massive_telemetry(num_rows=100000, filename="tower_telemetry_100k.csv"):
    print(f"Generating {num_rows:,} rows of synthetic network telemetry...")
    start_timer = time.time()
    
    # 1. Set up Base Timestamps (100ms intervals)
    base_time = int(time.time())
    timestamps = np.arange(base_time, base_time + num_rows)
    
    # 2. Assign Towers (Alternating between A and B)
    towers = np.where(np.arange(num_rows) % 2 == 0, 'TOWER_A', 'TOWER_B')
    
    # 3. Simulate Network Traffic with NumPy (Vectorized for extreme speed)
    # TOWER A is the busy one, TOWER B is the quiet backup
    is_tower_a = towers == 'TOWER_A'
    
    # Base users: Tower A (150-500), Tower B (50-150)
    users = np.zeros(num_rows, dtype=int)
    users[is_tower_a] = np.random.randint(150, 500, size=is_tower_a.sum())
    users[~is_tower_a] = np.random.randint(50, 150, size=(~is_tower_a).sum())
    
    # Inject the "Traffic Jam" anomaly into the last 20,000 rows for Tower A
    anomaly_mask = is_tower_a & (np.arange(num_rows) > (num_rows - 20000))
    # Exponential scaling for the anomaly
    surge_multiplier = np.linspace(1, 4, anomaly_mask.sum())
    users[anomaly_mask] = (users[anomaly_mask] * surge_multiplier).astype(int)
    
    # 4. Calculate Resource Utilization (PRB %) based on users + noise
    # Maxes out at 1.0 (100%)
    prb_util = (users / 1800.0) + np.random.normal(0, 0.02, size=num_rows)
    prb_util = np.clip(prb_util, 0.05, 1.0) # Keep between 5% and 100%
    
    # 5. Calculate Throughput (Gbps)
    throughput = users * np.random.uniform(0.002, 0.005, size=num_rows)
    
    # 6. Assemble the DataFrame
    df = pd.DataFrame({
        'timestamp': timestamps,
        'tower_id': towers,
        'active_users': users,
        'resource_block_util': np.round(prb_util, 4),
        'backhaul_throughput_gbps': np.round(throughput, 3)
    })
    
    # 7. Save to CSV
    df.to_csv(filename, index=False)
    
    elapsed = time.time() - start_timer
    print(f"✅ Success! {num_rows:,} rows saved to '{filename}' in {elapsed:.2f} seconds.")

if __name__ == "__main__":
    generate_massive_telemetry(num_rows=100000)

Generating 100,000 rows of synthetic network telemetry...
✅ Success! 100,000 rows saved to 'tower_telemetry_100k.csv' in 0.15 seconds.
